# ETF Liquidity Ranker
**Swarm Investments — Quant Workflow #1**

This notebook:
1. Downloads OHLCV data for ~300 US ETFs via `yfinance`
2. Computes 30-day average daily dollar volume (ADV) per ticker
3. Ranks the most liquid ETFs within each category
4. Sends a formatted HTML email report via Gmail API

**Production runs** are handled automatically every Monday by GitHub Actions.  
Use this notebook for development, exploration, and manual triggers.

---
### Gmail Auth in Colab
You have two options:
- **Option A (easiest):** Upload `token.json` directly to this Colab session (Files panel → Upload). It will be available for the session only.
- **Option B (persistent):** Mount Google Drive, store `token.json` there, and point `GMAIL_TOKEN_PATH` at it.


In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q yfinance pandas google-auth google-auth-oauthlib google-api-python-client

In [ ]:
# ── (Optional) Mount Google Drive to access persistent token.json ─────────────
# Skip this cell if you uploaded token.json directly to the Colab session.

# from google.colab import drive
# drive.mount('/content/drive')
# GMAIL_TOKEN_PATH = '/content/drive/MyDrive/swarm-quant/token.json'

In [ ]:
import base64
import logging
import os
import time
from datetime import datetime, timedelta
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from pathlib import Path

import pandas as pd
import yfinance as yf
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)
print('Imports OK')

In [ ]:
# ── Configuration — edit these as needed ──────────────────────────────────────

LOOKBACK_DAYS   = 30
MIN_ADV_USD     = 10_000_000   # $10M floor
TOP_N           = 5
GMAIL_SCOPES    = ['https://www.googleapis.com/auth/gmail.send']

RECIPIENT_EMAIL = 'njoyce@swarminvestments.com'
GMAIL_TOKEN_PATH = Path('token.json')   # change if using Drive

print(f'Config: TOP_N={TOP_N}, ADV>=${MIN_ADV_USD/1e6:.0f}M, lookback={LOOKBACK_DAYS}d')

In [ ]:
# ── ETF Universe ──────────────────────────────────────────────────────────────

ETF_UNIVERSE = {
    'US Large Cap Equity':              ['SPY','IVV','VOO','QQQ','VTI','SPLG','RSP','SCHB','ITOT','IWB','SPYG','SPYV','IWD','IWF'],
    'US Mid Cap Equity':                ['MDY','IJH','VO','IWR','SCHM','SPMD','IVOO','MDYG','MDYV'],
    'US Small Cap Equity':              ['IWM','VB','IJR','SCHA','VTWO','VIOO','SLY','SLYG','SLYV','IWC'],
    'International Developed Equity':   ['EFA','VEA','IDEV','SCHF','SPDW','VGK','EWJ','IEFA','FEZ','EWG','EWU','EWC','EWA'],
    'Emerging Markets Equity':          ['EEM','VWO','IEMG','SCHE','SPEM','EWZ','MCHI','EWY','EWT','INDA','EEMV'],
    'US Government Bonds':              ['TLT','IEF','SHY','GOVT','VGIT','VGLT','BIL','SHV','TBT','EDV','TMF','TBF','SPTS','SPTL'],
    'US Corporate Bonds':               ['LQD','VCIT','IGIB','QLTA','USIG','VCLT','IGSB','FLOT','JAAA'],
    'High Yield Bonds':                 ['HYG','JNK','USHY','FALN','ANGL','SJNK','SHYG','HYS'],
    'TIPS / Inflation-Linked':          ['TIP','VTIP','STIP','SCHP','LTPZ','PBTP'],
    'Real Estate (REITs)':              ['VNQ','IYR','SCHH','RWR','USRT','XLRE','REM','MORT'],
    'Commodities — Gold':               ['GLD','IAU','GLDM','SGOL','BAR','PHYS'],
    'Commodities — Broad':              ['DBC','PDBC','GSG','DJP','USCI','BCI'],
    'Commodities — Energy (ETPs)':      ['USO','UCO','SCO','UNG','BOIL','KOLD'],
    'Energy Sector':                    ['XLE','VDE','FENY','IYE','OIH','XOP','FCG'],
    'Technology Sector':                ['XLK','VGT','FTEC','IYW','SMH','SOXX','IGV','HACK','CIBR','WCLD'],
    'Healthcare Sector':                ['XLV','VHT','FHLC','IYH','IBB','XBI','ARKG','PPH'],
    'Financials Sector':                ['XLF','VFH','FNCL','IYF','KRE','KBE','IAI','KBWB'],
    'Consumer — Staples & Discretionary': ['XLP','VDC','FSTA','XLY','VCR','FDIS','IYC'],
    'Industrials & Materials':          ['XLI','VIS','FIDU','XLB','VAW','IYJ','IYM'],
    'Utilities & Communications':       ['XLU','VPU','FUTY','IDU','XLC','VOX'],
    'Factor — Momentum':                ['MTUM','PDP','QMOM','IMOM','MMTM','FDMO'],
    'Factor — Value':                   ['VTV','IVE','RPV','FVAL','VLUE','AVLV','QVAL'],
    'Factor — Quality':                 ['QUAL','DGRW','DGRO','SPHQ','JQUA'],
    'Factor — Low Volatility':          ['USMV','SPLV','EFAV','EEMV','FDLO'],
    'Dividend Income':                  ['VYM','SCHD','DVY','HDV','SDY','VIG','NOBL'],
    'Volatility Products':              ['VXX','UVXY','SVXY','VIXY','VIXM'],
    'Leveraged Equity (US)':            ['TQQQ','SQQQ','SPXL','SPXS','UPRO','SPXU','SSO','SDS'],
    'Thematic & Innovation':            ['ARKK','ARKG','ARKF','ARKW','BOTZ','ROBO','ICLN','TAN','FAN','DRIV','LIT','BATT'],
}

all_tickers = sorted({t for v in ETF_UNIVERSE.values() for t in v})
print(f'Universe: {len(ETF_UNIVERSE)} categories, {len(all_tickers)} unique tickers')

In [ ]:
# ── Fetch OHLCV data ──────────────────────────────────────────────────────────

end   = datetime.today()
start = end - timedelta(days=90)

log.info(f'Downloading data for {len(all_tickers)} tickers...')
raw = yf.download(
    all_tickers,
    start=start.strftime('%Y-%m-%d'),
    end=end.strftime('%Y-%m-%d'),
    group_by='ticker',
    auto_adjust=True,
    progress=True,
    threads=True,
)
print('Download complete.')

In [ ]:
# ── Compute metrics ───────────────────────────────────────────────────────────

records = []
for ticker in all_tickers:
    try:
        if isinstance(raw.columns, pd.MultiIndex):
            if ticker in raw.columns.get_level_values(0):
                df = raw[ticker]
            else:
                df = raw.xs(ticker, level=1, axis=1)
        else:
            df = raw

        df = df[['Close', 'Volume']].dropna().tail(LOOKBACK_DAYS)
        if len(df) < 10:
            continue

        n          = len(df)
        avg_vol    = float(df['Volume'].mean())
        avg_price  = float(df['Close'].mean())
        last_price = float(df['Close'].iloc[-1])
        adv_usd    = avg_vol * avg_price
        week_ret   = (df['Close'].iloc[-1] / df['Close'].iloc[max(0, n-5)]  - 1) * 100
        month_ret  = (df['Close'].iloc[-1] / df['Close'].iloc[0] - 1) * 100

        records.append({
            'ticker':     ticker,
            'avg_volume': avg_vol,
            'last_price': last_price,
            'adv_usd':    adv_usd,
            'week_ret':   float(week_ret),
            'month_ret':  float(month_ret),
        })
    except Exception as e:
        pass

df_all = pd.DataFrame(records)
print(f'{len(df_all)} tickers with valid data')
df_all.sort_values('adv_usd', ascending=False).head(10)

In [ ]:
# ── Fetch ETF names (only for tickers passing ADV floor) ──────────────────────

qualified = df_all.loc[df_all['adv_usd'] >= MIN_ADV_USD, 'ticker'].tolist()
print(f'{len(qualified)} tickers pass ${MIN_ADV_USD/1e6:.0f}M ADV floor. Fetching names...')

name_map = {}
for i, t in enumerate(qualified):
    try:
        info = yf.Ticker(t).info
        name_map[t] = info.get('shortName') or info.get('longName') or t
    except:
        name_map[t] = t
    if i % 25 == 24:
        time.sleep(1)

df_all['name'] = df_all['ticker'].map(name_map).fillna(df_all['ticker'])
print('Names fetched.')

In [ ]:
# ── Rank by category ──────────────────────────────────────────────────────────

ranked = {}
for category, tickers in ETF_UNIVERSE.items():
    cat = df_all[df_all['ticker'].isin(tickers) & (df_all['adv_usd'] >= MIN_ADV_USD)].copy()
    if cat.empty:
        continue
    cat = cat.sort_values('adv_usd', ascending=False).head(TOP_N).reset_index(drop=True)
    cat['rank'] = cat.index + 1
    ranked[category] = cat

print(f'Ranked {len(ranked)} categories')

# Preview one category
ranked['US Large Cap Equity'][['rank','ticker','name','last_price','adv_usd','week_ret','month_ret']]

In [ ]:
# ── Build HTML email (import helper from scripts/) ────────────────────────────
# If running in Colab, add the scripts directory to the path.

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'scripts'))

from etf_liquidity_ranker import build_email_html

html = build_email_html(ranked)
print(f'HTML built: {len(html):,} chars')

# Preview in notebook
from IPython.display import HTML
HTML(html)

In [ ]:
# ── Send email via Gmail API ──────────────────────────────────────────────────
# Requires token.json in the current directory (upload via Files panel or Drive).

from etf_liquidity_ranker import get_gmail_service, send_email
import os
os.environ['GMAIL_TOKEN_PATH'] = str(GMAIL_TOKEN_PATH)

today   = datetime.now().strftime('%B %d, %Y')
service = get_gmail_service()
send_email(service, RECIPIENT_EMAIL, f'ETF Liquidity Report — {today}', html)
print('Email sent!')